In [1]:
import pandas as pd

df=pd.read_excel("../data/ParsOffensive.xlsx")
df.head()

,comment,label
0,الان این فیلتر چه بادیگاردی هم براش گذاشتن یک ...,Offensive
1,سلام علیک سید جان,Neutral
2,سگ پدر,Offensive
3,خار چشم دشمنان ایران اسلامی سید عزیز,Neutral
4,سلام علیک بر سید عزیز,Neutral


In [2]:
import hazm

normalizer=hazm.Normalizer()
word_tokenizer=hazm.WordTokenizer()
lemmatizer=hazm.Lemmatizer()

def clean_stopword(stopword):
	return stopword.replace("آ", "ا")

stopwords=hazm.utils.stopwords_list()
stopwords=list(map(clean_stopword, stopwords))


for word in ["عالی", "خوب", "خوبی", "بی"]: stopwords.remove(word)

In [3]:
import string
import emoji
import numpy as np

def remove_chars(text, chars):
	for char in chars: text=text.replace(char, " ")
	return text

def preprocess(text):
	if not isinstance(text, str): return np.nan

	text=text.lower()
	text=remove_chars(text, string.punctuation+"؟،")

	text=emoji.replace_emoji(text, replace=" ")

	normalized_txt=normalizer.normalize(text).replace("آ", "ا")
	normalized_txt=remove_chars(normalized_txt, "۰۱۲۳۴۵۶۷۸۹\u200c")
	
	tokens=word_tokenizer.tokenize(normalized_txt)

	tokens=list(map(
		lambda token: lemmatizer.lemmatize(token).split("#")[0],
		tokens
	))

	tokens=list(filter(
		lambda token: token not in stopwords,
		tokens
	))

	if len(tokens)==0: return np.nan

	return " ".join(tokens)


df.comment=df.comment.apply(preprocess)
df.head()

,comment,label
0,الان فیلتر بادیگاردی براش گذاشت وقت لال نشه,Offensive
1,سلام علیک سید جان,Neutral
2,سگ پدر,Offensive
3,خار چشم دشمن ایران اسلامی سید عزیز,Neutral
4,سلام علیک سید عزیز,Neutral


In [4]:
print(df.isnull().sum())
df.dropna(inplace=True)
print(df.isnull().sum())
df.label.value_counts(normalize=True)

comment    170
label        0
dtype: int64
comment    0
label      0
dtype: int64


label
Neutral      0.628343
Offensive    0.371657
Name: proportion, dtype: float64

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer=TfidfVectorizer(
	ngram_range=(1, 2)
)

X=vectorizer.fit_transform(df.comment)

vectorizer.get_feature_names_out()

array(['npt', 'npt تولید', 'إسرائیلی', ..., 'یکیو عوض', 'یکیو یارست',
       'ییاد'], shape=(49810,), dtype=object)

In [6]:
import warnings
warnings.filterwarnings("ignore")

LABELS_DICT={"Neutral": 0, "Offensive": 1}
y=df.label.replace(LABELS_DICT, inplace=True).astype(int)
y

0       1
1       0
2       1
3       0
4       0
       ..
8428    0
8429    1
8430    1
8431    0
8432    0
Name: label, Length: 8263, dtype: int64

In [7]:
from sklearn.svm import LinearSVC

model=LinearSVC(class_weight="balanced")
model.fit(X, y)

,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

# Test

In [8]:
texts=[
    "خیلی مزخرف بود",
    "خیلی بی‌شعوری",
    "خیلی عالی بود",
    "واقعا جالب بود"
]

texts=list(map(preprocess, texts))

vecs=vectorizer.transform(texts)
predicts=model.predict(vecs)
print(predicts)
model._predict_proba_lr(vecs)

[1 1 0 0]


array([[0.24135965, 0.75864035],
       [0.27878422, 0.72121578],
       [0.71644883, 0.28355117],
       [0.74247531, 0.25752469]])